This notebook allows you to run ACOLITE processing on Google Colab. The required dependencies are installed and the latest version is cloned from GitHub (https://github.com/acolite/acolite).

In this example, Sentinel-2 scenes are retrieved from Google Cloud Storage/Earth Engine buckets, processed and stored on your Google Drive. If you set up your credentials, scenes from CDSE and EarthExplorer can also be downloaded using the API in ACOLITE. Ancillary data can be retrieved if EarthData credentials are set up, but are not essential for this example.

Author: Quinten Vanhellemont, RBINS, quinten.vanhellemont@naturalsciences.be

The next cell sets up the ACOLITE dependencies in Colab if they are not already there. This is specifical to Colab and you will need to set up an ACOLITE environment if you would want to run it locally.

In [ ]:
# general imports
import os, sys

# install dependencies for acolite
# for colab: assume if netCDF4 is installed the other dependencies are as well
try:
  import netCDF4
except:
  !pip install numpy matplotlib scipy gdal pyproj scikit-image pyhdf pyresample netcdf4 h5py requests pygrib cartopy

In [ ]:
# clone acolite repo if it does not exist
if not os.path.exists('acolite'):
  !git clone --depth 1 https://github.com/acolite/acolite

In [ ]:
# add acolite to path and import
sys.path.append('acolite')
import acolite as ac

The next cell configures the use of Google Cloud Storage to retrieve S2 data, and otherwise imports data from the credentials.txt file. Both of these are specific to Colab. For running locally you would ideally set up those credentials in a .netrc file.

In [ ]:
# auth is needed for retrieving S2 data from GCS
# this will ask you to give permissions to this notebook to access your Google Drive
use_gcs = True
if use_gcs:
  from google.colab import auth
  auth.authenticate_user()

## otherwise we will use the CDSE API for download
## and credentials need to be imported
else:
  ## we will use the CDSE API for download
  ## and credentials need to be imported
  ## load credentials and set as env variables
  if os.path.exists('credentials.txt'):
    credentials = ac.shared.import_config('credentials.txt')
    for k in credentials:
      if len(credentials[k]) > 0:
        os.environ[k] = credentials[k]
        print('Set {} as environment variable'.format(k))
      else:
        print('{} empty in credentials.txt'.format(k))
  else:
    print('Error: credentials.txt not found. Please upload to current runtime.')

In [ ]:
## location and date of interest
## you can use the Copernicus browser to find images
## https://browser.dataspace.copernicus.eu/
region_name = 'SanServolo'
station_lon = 12.3573
station_lat = 45.4188
date = '2026-07-05'
date = '2026-07-11'

In [ ]:
## use ACOLITE API to query CDSE
urls, scenes = ac.api.cdse.query(roi = 'POINT ({} {})'.format(station_lon, station_lat),
                                 collection = "SENTINEL-2", start_date = date, end_date = date)
## print the results
for scene in scenes:
  print(scene)

In [ ]:
# make input directory
if not os.path.exists('Input/'):
  os.makedirs('Input/')

local_scenes = []
for scene in scenes:
  local_scene = 'Input/{}'.format(scene)
  if os.path.exists(local_scene):
    print('We have {}'.format(local_scene))
    local_scenes.append(local_scene)
    continue

  ## use GCS to retrieve the S2 scene
  if use_gcs:
    ## figure out scene location on GCS
    if scene.startswith('S2'):
      tile = scene.split('_')[-2]
      zone = tile[1:3]
      t1 = tile[3]
      t2 = tile[4:6]
      gs_url = 'gs://gcp-public-data-sentinel-2/tiles/{}/{}/{}/{}'.format(zone, t1, t2, scene)
      print('Scene url: {}'.format(gs_url))
    else:
      print('{} not supported'.format(scene))

    # let's get the scene from GCS
    if not os.path.exists(local_scene):
      !gcloud storage --verbosity error cp -r $gs_url Input/

  ## use ACOLITE API to download the scenes from CDSE
  if not os.path.exists(local_scene):
    local_scene = ac.api.cdse.download(urls, scenes = scenes, output = 'Input/')

  ## add to local scenes list
  if os.path.exists(local_scene):
    print('Downloaded {}'.format(local_scene))
    local_scenes.append(local_scene)

The next cell sets up a Python dictionary with typical settings for processing ACOLITE/DSF for a region of interest (here a square box centred on a location).

In [ ]:
# acolite settings
settings = {## basic input and output settings
            ## local_scenes list is used as returned from the API download function
            'inputfile': local_scenes,
            'output': 'Output/DSF',
            'l2w_parameters': ['Rrs_*'],
            'l2w_rgb_keys': ['Rrs'],
            'rgb_Rrs': True,

            ## set up region of interest centred on lon/lat, with box size in km
            'region_name': region_name,
            'station_lon': station_lon,
            'station_lat': station_lat,
            'station_box_size': 7,
            'station_box_units': 'km',

            ## disable ancillary data retrieval, set to True if EarthData credentials are set
            'ancillary_data': False,

            ## use DSF processing
            'atmospheric_correction_method': 'dark_spectrum',

            ## disable negatives masking
            'l2w_mask_negative_rhow': False,

            ## delete L1R output NetCDF
            'l1r_delete_netcdf': True,
            ## don't output L1R RGBs
            'l1r_rgb_keys': [],
            ## delete ACOLITE text outputs
            'delete_acolite_run_text_files': True,
            }

The next cell passes the settings dictionary to ACOLITE and runs the processing. Note that the first run will be slightly slower, as the LUTs for the current sensor will have to be downloaded.

In [ ]:
## run acolite with settings dict
ret = ac.acolite.acolite_run(settings)

The next cell creates a new settings dictionary, this time activating the SWIR based glint correction. It is immediately passed to ACOLITE for processing.

In [ ]:
# acolite settings
settings = {## basic input and output settings
            ## local_scenes list is used as returned from the API download function
            'inputfile': local_scenes,
            'output': 'Output/DSF+GC',
            'l2w_parameters': ['Rrs_*'],
            'l2w_rgb_keys': ['Rrs'],
            'rgb_Rrs': True,

            ## set up region of interest centred on lon/lat, with box size in km
            'region_name': region_name,
            'station_lon': station_lon,
            'station_lat': station_lat,
            'station_box_size': 7,
            'station_box_units': 'km',

            ## disable ancillary data retrieval, set to True if EarthData credentials are set
            'ancillary_data': False,

            ## use DSF processing
            'atmospheric_correction_method': 'dark_spectrum',

            ## add the following to enable SWIR based glint correction
            'dsf_residual_glint_correction': True,
            'glint_mask_rhos_threshold': 0.10,

            ## disable negatives masking
            'l2w_mask_negative_rhow': False,

            ## delete L1R output NetCDF
            'l1r_delete_netcdf': True,
            ## don't output L1R RGBs
            'l1r_rgb_keys': [],
            ## delete ACOLITE text outputs
            'delete_acolite_run_text_files': True,
            }

## run acolite with settings dict
ret = ac.acolite.acolite_run(settings)

The next cell sets up RAdCor processing, with an automatic buffer applied to account for the PSF radius.

In [ ]:
# acolite settings
settings = {## basic input and output settings
            ## local_scenes list is used as returned from the API download function
            'inputfile': local_scenes,
            'output': 'Output/RAdCor',
            'l2w_parameters': ['Rrs_*'],
            'l2w_rgb_keys': ['Rrs'],
            'rgb_Rrs': True,

            ## set up region of interest centred on lon/lat, with box size in km
            'region_name': region_name,
            'station_lon': station_lon,
            'station_lat': station_lat,
            'station_box_size': 7,
            'station_box_units': 'km',

            ## disable ancillary data retrieval, set to True if EarthData credentials are set
            'ancillary_data': False,

            ## use RAdCor processing
            'atmospheric_correction_method': 'radcor',

            ## add a buffer and crop to centre to avoid masked data around the scene edges
            'limit_buffer': 5,
            'limit_buffer_units': 'km',
            'radcor_crop_centre': True,

            ## disable negatives masking
            'l2w_mask_negative_rhow': False,

            ## delete L1R output NetCDF
            'l1r_delete_netcdf': True,
            ## don't output L1R RGBs
            'l1r_rgb_keys': [],
            ## delete ACOLITE text outputs
            'delete_acolite_run_text_files': True,
            }

## run acolite with settings dict
ret = ac.acolite.acolite_run(settings)

The next cell also runs RAdCor, with the SWIR based glint correction activated.

In [ ]:
# acolite settings
settings = {## basic input and output settings
            ## local_scenes list is used as returned from the API download function
            'inputfile': local_scenes,
            'output': 'Output/RAdCor+GC',
            'l2w_parameters': ['Rrs_*'],
            'l2w_rgb_keys': ['Rrs'],
            'rgb_Rrs': True,

            ## set up region of interest centred on lon/lat, with box size in km
            'region_name': region_name,
            'station_lon': station_lon,
            'station_lat': station_lat,
            'station_box_size': 7,
            'station_box_units': 'km',

            ## disable ancillary data retrieval, set to True if EarthData credentials are set
            'ancillary_data': False,

            ## use RAdCor processing
            'atmospheric_correction_method': 'radcor',

            ## add a buffer and crop to centre to avoid masked data around the scene edges
            'limit_buffer': 5,
            'limit_buffer_units': 'km',
            'radcor_crop_centre': True,

            ## add the following to enable SWIR based glint correction
            'dsf_residual_glint_correction': True,
            'glint_mask_rhos_threshold': 0.10,

            ## disable negatives masking
            'l2w_mask_negative_rhow': False,

            ## delete L1R output NetCDF
            'l1r_delete_netcdf': True,
            ## don't output L1R RGBs
            'l1r_rgb_keys': [],
            ## delete ACOLITE text outputs
            'delete_acolite_run_text_files': True,
            }

## run acolite with settings dict
ret = ac.acolite.acolite_run(settings)

In [ ]:
## imports for plotting
import matplotlib.pyplot as plt
import numpy as np
import glob

## extraction location
ext_lon, ext_lat = 12.35489, 45.44826
#ext_lon, ext_lat = 12.38, 45.40

## find NetCDF
ncf = glob.glob('Output/DSF/*_L2R.nc')[0]

## load data
gatts = ac.shared.nc_gatts(ncf)
title = '{} {}'.format(gatts['sensor'].replace('_', '/'), gatts['isodate'][0:19])

lat = ac.shared.nc_data(ncf, 'lat')
lon = ac.shared.nc_data(ncf, 'lon')
datasets = ac.shared.nc_datasets(ncf)

stack = []
waves = []
for ds in datasets:
  if ds.startswith('rhos_'):
      waves.append(float(ds[ds.find('_')+1:]))
      stack.append(ac.shared.nc_data(ncf, ds))
stack = np.asarray(stack)
waves = np.asarray(waves)
rgb = ac.map.rgb.stack(waves, stack,  rgb_max = [0.15] * 3,)

fig, ax = plt.subplots()
plt.xlabel('Longitude (°E)')
plt.ylabel('Latitude (°N)')
plt.title(title)

mesh = plt.pcolormesh(lon, lat, rgb)
plt.plot(ext_lon, ext_lat, 'x', color = 'Red')
plt.savefig('Output/RGB.png', dpi = 300, bbox_inches = 'tight', facecolor = 'white')

In [ ]:
for processor in ['DSF', 'DSF+GC', 'RAdCor', 'RAdCor+GC']:
  ncf = glob.glob('Output/{}/*_L2W.nc'.format(processor))[0]
  ext = ac.shared.nc_extract_point(ncf, ext_lon, ext_lat, box_size = 11)

  title = '{} {}'.format(ext['gatts']['sensor'].replace('_', '/'), ext['gatts']['isodate'][0:19])
  mean = np.asarray([np.nanmean(ext['data'][ds]) for ds in ext['Rrs_datasets']])
  std = np.asarray([np.nanstd(ext['data'][ds]) for ds in ext['Rrs_datasets']])
  plt.errorbar(ext['Rrs_wave'], mean, marker = '.', linestyle = '--', yerr = std, label = processor)

plt.xlabel('Wavelength (nm)')
plt.ylabel(r'Rrs ($sr^{-1}$)')

plt.ylim(-0.002, 0.042)
plt.xlim(400, 900)

plt.legend()
plt.title(title)

plt.savefig('Output/Rrs_comparison.png', dpi = 300, bbox_inches = 'tight', facecolor = 'white')
